In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [ ]:
# Q16: Service combination with lowest churn
combos = df.groupby(['InternetService', 'TechSupport'])['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index()
combos.columns = ['InternetService', 'TechSupport', 'churn_rate']
combos.sort_values('churn_rate')

In [ ]:
# Q17: Services most strongly associated with churn reduction
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

results = []
for service in service_cols:
    churn_with    = round((df[df[service] == 'Yes']['Churn'] == 'Yes').sum() /
                    len(df[df[service] == 'Yes']) * 100, 2)
    churn_without = round((df[df[service] == 'No']['Churn'] == 'Yes').sum() /
                    len(df[df[service] == 'No']) * 100, 2)
    reduction     = round(churn_without - churn_with, 2)

    results.append({
        'service'        : service,
        'churn_with'     : churn_with,
        'churn_without'  : churn_without,
        'churn_reduction': reduction
    })

pd.DataFrame(results).sort_values('churn_reduction', ascending=False)

In [ ]:
# Q18: Build a Service Adoption Score and analyze its relationship with churn
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['adoption_score'] = df[service_cols].apply(
    lambda row: (row == 'Yes').sum(), axis=1
)

df.groupby('adoption_score')['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index().rename(columns={'Churn': 'churn_rate'})

In [ ]:
# Q19: Fiber optic, no tech support, no online security — churn rate
high_risk = df[
    (df['InternetService'] == 'Fiber optic') &
    (df['TechSupport'] == 'No') &
    (df['OnlineSecurity'] == 'No')
]

print(f'Customers matching criteria: {len(high_risk)}')
print(f'Churned: {(high_risk["Churn"] == "Yes").sum()}')
print(f'Churn rate: {round((high_risk["Churn"] == "Yes").sum() / len(high_risk) * 100, 2)}%')

In [ ]:
# Q20: Most profitable service bundle
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['bundle'] = df[service_cols].apply(
    lambda row: '+'.join([col for col in service_cols if row[col] == 'Yes']), axis=1
)

df['bundle'] = df['bundle'].replace('', 'No Services')

bundle_analysis = df.groupby('bundle').agg(
    customer_count = ('customerID', 'count'),
    avg_monthly    = ('MonthlyCharges', lambda x: round(x.mean(), 2)),
    churn_rate     = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)),
    avg_clv        = ('TotalCharges', lambda x: round(x.mean(), 2))
).reset_index()

bundle_analysis = bundle_analysis[bundle_analysis['customer_count'] >= 30]

bundle_analysis.sort_values('avg_clv', ascending=False).head(10)